# 04 · Target redesign as a robustness experiment

This stage does **not** claim a replacement production model. It reconstructs a stricter evaluation dataset to ask how much farther-horizon signal remains after separating reference, observation, lead, and outcome windows.

In [ ]:
from deposit_runoff.config import load_config
from deposit_runoff.demo_data import make_demo_daily_panel
from deposit_runoff.targets import build_v2_robustness_dataset

cfg = load_config('../config/public.yaml')
d = cfg['data']
daily = make_demo_daily_panel(n_accounts=d['n_accounts'], start_date=str(d['start_date']), end_date=str(d['end_date']), seed=cfg['seed'])
snapshots = [*map(str, cfg['snapshots']['train']), str(cfg['snapshots']['validation']), str(cfg['snapshots']['test'])]
v2 = build_v2_robustness_dataset(daily, snapshots, **cfg['v2_robustness'])
v2.groupby('snapshot_date')['runoff_flag'].agg(['size','mean'])

## Design

`reference (t-59:t-30) → observation (t-29:t) → lead (t+1:t+14) → outcome (t+15:t+44)`

The reference is deliberately separated from the observation window. Eligibility screens accounts that have already deteriorated materially at time *t*. The lead-period screen uses the minimum 7-day rolling average rather than a one-day minimum so transient dips do not automatically remove an account.

## Why the lead screen is explicitly labeled a robustness design

The lead-window censor uses behavior after scoring time. Therefore it cannot define the population of a deployable scorer at time *t*. Its purpose is conditional evaluation: remove near-term deterioration and test a harder, farther-horizon cohort.

## Where the original work stopped

The redesigned multi-snapshot dataset was constructed and validated. Training a new model directly on this target is a natural next experiment, but no revised-model performance is claimed in this case study.